# 01 — Explorative Datenanalyse & Datenqualität

**Ziel dieses Notebooks:** Den Disaster-Tweets-Datensatz systematisch und
datengetrieben kennenlernen, bevor Modellierungsentscheidungen getroffen
werden. Keine Annahmen aus Vorwissen über den Datensatz.

**Leitfragen:**
- Wie ist die Datenstruktur beschaffen (fehlende Werte, Duplikate)?
- Wie ist die Zielvariable verteilt?
- Wie unterscheiden sich Tweets beider Klassen (Länge, Wortwahl)?
- Wie groß/informativ ist das Vokabular?
- Gibt es Label-Rauschen oder strukturelles Rauschen im Text?
- Sind `keyword`/`location` nutzbar?
- Wie gut ist eine triviale Baseline (untere Leistungsgrenze)?

**Output:** Plots in `reports/figures/`, Tabellen in `reports/tables/`,
Erkenntnisse fortlaufend in `reports/findings.md`.

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Store44-Style aktivieren, Daten laden
# =============================================================================

import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

# Daten laden (Kaggle-Vollversion, siehe reports/findings.md fuer Begruendung)
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
train.head()

In [ ]:
# =============================================================================
# Zelle 03 – Struktur-Check: Datentypen, fehlende Werte, Duplikate & Konflikte
# =============================================================================

print("=== Datentypen ===")
print(train.dtypes)

print("\n=== Fehlende Werte (absolut & relativ) ===")
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_summary)
missing_summary.to_csv("../reports/tables/01_missing_values.csv")

print("\n=== Duplikate & Label-Konflikte ===")

# Flag 1: Text kommt mit unterschiedlichen Labels vor (echtes Label-Rauschen)
label_variation = train.groupby("text")["target"].nunique()
conflicting_texts = label_variation[label_variation > 1].index
train["is_label_conflict"] = train["text"].isin(conflicting_texts)

# Flag 2: exaktes Duplikat (Text+Label identisch, weiteres Vorkommen)
train["is_exact_duplicate"] = train.duplicated(subset=["text", "target"], keep=False) & ~train["is_label_conflict"]

print("Texte mit widerspruechlichen Labels:", len(conflicting_texts))
print("Davon betroffene Zeilen:", train["is_label_conflict"].sum())
print("Harmlose exakte Duplikat-Zeilen (Text+Label gleich):", train["is_exact_duplicate"].sum())

# Export aller betroffenen Zeilen zur manuellen Durchsicht
review_df = train[train["is_label_conflict"] | train["is_exact_duplicate"]].sort_values("text")
review_df.to_csv("../reports/tables/01_duplicates_conflicts_review.csv", index=False)
print(f"\n{len(review_df)} Zeilen zur Durchsicht exportiert: reports/tables/01_duplicates_conflicts_review.csv")

In [ ]:
# =============================================================================
# Zelle 03b – Erweiterte Duplikat-/Konflikt-Pruefung auf text_clean-Basis
# =============================================================================
# Notebook-01-Bereinigung (Zelle 03) prueft nur auf Roh-Text. URL-Entfernung,
# Lowercase und NUM-Platzhalter (Notebook 02) erzeugen weitere Duplikate/
# Konflikte, die hier bereits identifiziert werden (spaetere Bereinigung
# erfolgt in Notebook 02 nach text_clean-Erzeugung).

import re
import html as html_module

def clean_text_preview(text):
    """Vereinfachte Vorschau von text_clean (gleiche Logik wie Notebook 02)."""
    text = html_module.unescape(str(text))
    text = re.sub(r"http[s]?://\S+", "", text)
    text = re.sub(r"[\x80-\xff]", "", text)
    text = re.sub(r"\d+(?:[.,]\d+)*", "NUM", text)
    text = text.lower()
    return re.sub(r"\s+", " ", text).strip()

# text_clean auf aktuellem train (noch nicht bereinigt) erzeugen
train["text_clean_preview"] = train["text"].apply(clean_text_preview)

# Duplikat-Gruppen auf text_clean-Basis
dupes_clean = train[train.duplicated(subset=["text_clean_preview"], keep=False)].copy()
conflict_groups_clean = dupes_clean.groupby("text_clean_preview")["target"].nunique()
conflict_texts_clean = conflict_groups_clean[conflict_groups_clean > 1].index
harmless_texts_clean = conflict_groups_clean[conflict_groups_clean == 1].index

print("=== Duplikate nach text_clean-Normalisierung (URL/Lowercase/NUM) ===")
print(f"Betroffene Zeilen gesamt:                {len(dupes_clean)}")
print(f"Gruppen gesamt:                          {dupes_clean['text_clean_preview'].nunique()}")
print(f"Konflikt-Gruppen (widersprueche Labels): {len(conflict_texts_clean)}")
print(f"Harmlose Duplikat-Gruppen:               {len(harmless_texts_clean)}")

# Ursachen-Analyse: wie viele Konflikte entstehen durch URL?
no_url_conflicts = sum(
    1 for ct in conflict_texts_clean
    if not dupes_clean[dupes_clean["text_clean_preview"] == ct]["text"]
    .str.contains(r"http", na=False).any()
)
url_conflicts = len(conflict_texts_clean) - no_url_conflicts
print(f"\nKonflikt-Gruppen durch URL-Normalisierung: {url_conflicts} ({url_conflicts/len(conflict_texts_clean)*100:.1f}%)")
print(f"Konflikt-Gruppen ohne URL (echtes Rauschen): {no_url_conflicts} ({no_url_conflicts/len(conflict_texts_clean)*100:.1f}%)")

# Export zur Dokumentation
review_clean = dupes_clean[dupes_clean["text_clean_preview"].isin(conflict_texts_clean)][
    ["id", "text", "text_clean_preview", "target"]
].sort_values("text_clean_preview")
review_clean.to_csv("../reports/tables/01_label_conflicts_after_cleaning.csv", index=False)
print(f"\nExportiert: reports/tables/01_label_conflicts_after_cleaning.csv")

## Erkenntnis: Struktur, fehlende Werte, Duplikate & Konflikte

**Fehlende Werte:**
- `keyword`: 61 fehlend (0,80 %) — vernachlässigbar
- `location`: 2.533 fehlend (33,27 %) — erheblich; zusätzlich vermutlich
  geringe inhaltliche Zuverlässigkeit auch bei vorhandenen Werten (Twitter-
  Freitextfeld), naehere Pruefung folgt bei der keyword/location-Analyse
  spaeter in diesem Notebook

**Duplikate & Label-Konflikte:**
- **18 Texte (55 Zeilen)** mit widerspruechlichen Labels — echtes
  Label-Rauschen, begrenzt den theoretisch erreichbaren Bayes-Fehler
  unabhaengig vom Modell
- **124 Zeilen** harmlose exakte Duplikate (Text+Label identisch) — Risiko:
  Data Leakage bei K-Fold CV, falls identische Tweets in Trainings- und
  Validierungs-Fold landen

**Entscheidung:** Beide Kategorien werden vor der Modellierung entfernt
(Nachvollziehbarkeit: `reports/tables/01_duplicates_conflicts_review.csv`).
Bereinigung erfolgt am Ende dieses Notebooks als eigene Zelle, Ergebnis wird
als `data/processed/train_clean.csv` gespeichert.

## Erweiterung: Duplikate/Konflikte nach text_clean-Normalisierung

Nach URL-Entfernung, Lowercase und NUM-Platzhalter entstehen
zusaetzliche Duplikat-Gruppen, die auf Roh-Text-Ebene (Zelle 03)
nicht erkennbar waren:

| | Gruppen | Zeilen |
|---|---|---|
| Konflikt-Gruppen (widersprueche Labels) | 70 | 253 |
| Davon URL-bedingt | 54 (77%) | 201 (79%) |
| Davon echtes Label-Rauschen (kein URL) | 16 (23%) | 52 (21%) |

Alle Zeilen haben ein gueltiges Label (0 oder 1) — keine fehlenden
Werte. Die 16 nicht-URL-bedingten Konflikt-Gruppen (z. B. "To fight
bioterrorism sir." 4x mit wechselnden Labels) sind genuine
Label-Inkonsistenzen, die auch nach Normalisierung bestehen bleiben.

**Konsequenz:** Bereinigung erfolgt in Notebook 02 auf text_clean-
Basis (zusaetzlich zur Roh-Text-Bereinigung aus Zelle 03). Export:
reports/tables/01_label_conflicts_after_cleaning.csv

In [ ]:
# =============================================================================
# Zelle 05 – Klassenbalance
# =============================================================================

class_counts = train["target"].value_counts().sort_index()
class_pct = (class_counts / len(train) * 100).round(1)

print("Klassenverteilung (absolut):\n", class_counts)
print("\nKlassenverteilung (%):\n", class_pct)

# Export
class_summary = pd.DataFrame({"count": class_counts, "pct": class_pct})
class_summary.to_csv("../reports/tables/01_class_balance.csv")

# Plot im Store44-Stil
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    ["Kein Disaster (0)", "Disaster (1)"],
    class_counts.values,
    color=[COLOR_CLASS_0, COLOR_CLASS_1]
)
ax.set_ylabel("Anzahl Tweets")
ax.set_title("Klassenverteilung")

# Werte ueber den Balken anzeigen
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"{count}\n({pct}%)", ha="center", color=COLOR_TEXT_MUTED if False else "white")

# Y-Achse mit zusaetzlichem Spielraum nach oben (15% statt Standard)
ax.set_ylim(0, class_counts.max() * 1.15)

save_figure(fig, "../reports/figures/01_eda_class_balance.png")
plt.show()

## Erkenntnis: Klassenverteilung

57,0 % (4.342) "kein Disaster", 43,0 % (3.271) "Disaster" — moderates
Ungleichgewicht, keine extreme Schieflage.

**Konsequenz fuer Modellierung:**
- F1-Score (statt reiner Accuracy) als Hauptmetrik bleibt gerechtfertigt,
  aber bei diesem Grad an Imbalance ist Accuracy nicht grob irrefuehrend
  (ein Dummy-Klassifikator "immer Klasse 0" laege bei ~57 % Accuracy —
  exakter Wert folgt in der Baseline-Zelle)
- Class Weighting o. ae. vermutlich nicht zwingend erforderlich bei dieser
  moderaten Verteilung; relevant waere es eher ab ca. 80/20 oder staerker
- Stratified K-Fold CV (ohnehin Pflicht) stellt sicher, dass jeder Fold
  das gleiche Verhaeltnis beibehaelt

In [ ]:
# =============================================================================
# Zelle 07 – Textlaengen pro Klasse (Zeichen & Woerter)
# =============================================================================

train["char_count"] = train["text"].str.len()
train["word_count"] = train["text"].str.split().str.len()

length_stats = train.groupby("target")[["char_count", "word_count"]].describe()
print(length_stats)

length_stats.to_csv("../reports/tables/01_text_length_stats.csv")

# Plot: Verteilung Zeichenlaenge pro Klasse
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for target_val, color, label in [(0, COLOR_CLASS_0, "Kein Disaster"), (1, COLOR_CLASS_1, "Disaster")]:
    subset = train[train["target"] == target_val]
    axes[0].hist(subset["char_count"], bins=30, alpha=0.6, color=color, label=label)
    axes[1].hist(subset["word_count"], bins=20, alpha=0.6, color=color, label=label)

axes[0].set_title("Zeichenanzahl pro Tweet")
axes[0].set_xlabel("Zeichen")
axes[0].set_ylabel("Anzahl Tweets")
axes[0].legend()

axes[1].set_title("Wortanzahl pro Tweet")
axes[1].set_xlabel("Woerter")
axes[1].legend()

fig.suptitle("Textlaengen-Verteilung nach Klasse")
fig.tight_layout()

save_figure(fig, "../reports/figures/01_eda_text_length.png")
plt.show()

## Erkenntnis: Textlaengen pro Klasse

Wortanzahl praktisch identisch zwischen den Klassen (14,7 vs. 15,2 Woerter).
Der scheinbare Unterschied bei der Zeichenanzahl (95,7 vs. 108,1, Mittelwert)
ist NICHT auf generell laengere Formulierungen zurueckzufuehren: in 12 von 16
Laengen-Bins liegt "Kein Disaster" sogar haeufiger vor. Treiber ist ein
einzelner, stark besetzter Bin nahe des alten Twitter-Zeichenlimits (~140),
in dem Disaster-Tweets ueberproportional vertreten sind.

**Ursache (geprueft):** Disaster-Tweets enthalten deutlich haeufiger URLs
(66,4 % vs. 41,4 %) – vermutlich Verweise auf Nachrichtenartikel/automatisierte
Disaster-Bots. Der feste URL-Anteil drueckt diese Tweets naeher ans Zeichenlimit.

**Konsequenz:** "URL vorhanden" (binaeres Feature) ist vermutlich informativer
als die reine Zeichenlaenge und wird in der Preprocessing-Phase als Kandidat
gefuehrt. Reine Laengen-Features (char_count) erscheinen dagegen wenig
aussagekraeftig fuer die Klassifikation.

In [ ]:
# =============================================================================
# Zelle 07b – URL-Praesenz als neue Spalte anhaengen + Vergleich nach Klasse
# =============================================================================

train["has_url"] = train["text"].str.contains(r"http[s]?://", regex=True)

url_by_class = (train.groupby("target")["has_url"].mean() * 100).round(1)
print("Anteil Tweets MIT URL, nach Klasse (%):")
print(url_by_class)

url_summary = pd.DataFrame({"pct_with_url": url_by_class})
url_summary.to_csv("../reports/tables/01_url_presence_by_class.csv")

# Plot: URL-Anteil pro Klasse
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    ["Kein Disaster (0)", "Disaster (1)"],
    url_by_class.values,
    color=[COLOR_CLASS_0, COLOR_CLASS_1]
)
ax.set_ylabel("Anteil Tweets mit URL (%)")
ax.set_title("URL-Praesenz nach Klasse")
ax.set_ylim(0, 100)

for bar, pct in zip(bars, url_by_class.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{pct}%", ha="center", color="white")

save_figure(fig, "../reports/figures/01_eda_url_presence.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 09 – Vokabulargroesse: waechst es noch mit der Korpusgroesse?
# =============================================================================

import re

def simple_tokenize(text):
    # Einfache Tokenisierung: lowercase, nur Wortzeichen, fuer EDA-Zwecke ausreichend
    return re.findall(r"\b\w+\b", text.lower())

# Vokabular je Klasse getrennt + gesamt
all_tokens = train["text"].apply(simple_tokenize)
vocab_overall = set(token for tokens in all_tokens for token in tokens)

vocab_class0 = set(token for tokens in all_tokens[train["target"] == 0] for token in tokens)
vocab_class1 = set(token for tokens in all_tokens[train["target"] == 1] for token in tokens)

print("Vokabulargroesse gesamt:", len(vocab_overall))
print("Vokabulargroesse Klasse 0 (kein Disaster):", len(vocab_class0))
print("Vokabulargroesse Klasse 1 (Disaster):", len(vocab_class1))
print("Ueberschneidung (gemeinsame Woerter):", len(vocab_class0 & vocab_class1))
print("Nur in Klasse 0:", len(vocab_class0 - vocab_class1))
print("Nur in Klasse 1:", len(vocab_class1 - vocab_class0))

# Heaps' Law: Vokabular-Wachstum vs. Korpusgroesse (zeigt, ob Vokabular "saettigt")
import numpy as np

sample_sizes = np.linspace(100, len(train), 30, dtype=int)
vocab_growth = []
seen_vocab = set()
shuffled_tokens = all_tokens.sample(frac=1, random_state=42).reset_index(drop=True)

cumulative_tokens = []
running_vocab = set()
for i, tokens in enumerate(shuffled_tokens):
    running_vocab.update(tokens)
    if (i + 1) in sample_sizes or i == len(shuffled_tokens) - 1:
        cumulative_tokens.append((i + 1, len(running_vocab)))

heaps_df = pd.DataFrame(cumulative_tokens, columns=["n_documents", "vocab_size"])
heaps_df.to_csv("../reports/tables/01_vocab_growth.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(heaps_df["n_documents"], heaps_df["vocab_size"], color=COLOR_GOLD, linewidth=2)
ax.set_xlabel("Anzahl Tweets (kumulativ)")
ax.set_ylabel("Vokabulargroesse (unique Tokens)")
ax.set_title("Vokabular-Wachstum (Heaps' Law)")

save_figure(fig, "../reports/figures/01_eda_vocab_growth.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 09b – Venn-Diagramm: Vokabular-Ueberschneidung zwischen den Klassen
# =============================================================================

import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))

# Zwei ueberlappende Kreise (gleich gross fuer klare Lesbarkeit, nicht
# flaechenproportional - reine Mengen-Visualisierung)
circle0 = patches.Circle((0.35, 0.5), 0.3, color=COLOR_CLASS_0, alpha=0.6)
circle1 = patches.Circle((0.65, 0.5), 0.3, color=COLOR_CLASS_1, alpha=0.6)
ax.add_patch(circle0)
ax.add_patch(circle1)

only_0 = len(vocab_class0 - vocab_class1)
only_1 = len(vocab_class1 - vocab_class0)
overlap = len(vocab_class0 & vocab_class1)

ax.text(0.22, 0.5, f"{only_0}", ha="center", va="center", fontsize=16, color="white")
ax.text(0.78, 0.5, f"{only_1}", ha="center", va="center", fontsize=16, color="white")
ax.text(0.5, 0.5, f"{overlap}", ha="center", va="center", fontsize=16, color="white")

ax.text(0.22, 0.85, "Nur Kein Disaster", ha="center", fontsize=11, color=COLOR_CLASS_0)
ax.text(0.78, 0.85, "Nur Disaster", ha="center", fontsize=11, color=COLOR_CLASS_1)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Vokabular-Ueberschneidung zwischen den Klassen")

save_figure(fig, "../reports/figures/01_eda_vocab_venn.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 09c – Vokabular-Ueberschneidung OHNE Stopwords (Vergleich)
# =============================================================================

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

vocab_class0_clean = vocab_class0 - stop_words
vocab_class1_clean = vocab_class1 - stop_words

overlap_clean = len(vocab_class0_clean & vocab_class1_clean)
only_0_clean = len(vocab_class0_clean - vocab_class1_clean)
only_1_clean = len(vocab_class1_clean - vocab_class0_clean)

print("=== Mit Stopwords (bisher) ===")
print(f"Ueberschneidung: {overlap} | Nur Klasse 0: {only_0} | Nur Klasse 1: {only_1}")

print("\n=== Ohne Stopwords ===")
print(f"Ueberschneidung: {overlap_clean} | Nur Klasse 0: {only_0_clean} | Nur Klasse 1: {only_1_clean}")

stopwords_in_overlap = len((vocab_class0 & vocab_class1) & stop_words)
print(f"\nAnteil Stopwords an der urspruenglichen Ueberschneidung: {stopwords_in_overlap}/{overlap} ({stopwords_in_overlap/overlap*100:.1f}%)")

In [ ]:
# =============================================================================
# Zelle 09b – Venn-Diagramm: Vokabular-Ueberschneidung (ohne Stopwords)
# =============================================================================

import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))

circle0 = patches.Circle((0.35, 0.5), 0.3, color=COLOR_CLASS_0, alpha=0.6)
circle1 = patches.Circle((0.65, 0.5), 0.3, color=COLOR_CLASS_1, alpha=0.6)
ax.add_patch(circle0)
ax.add_patch(circle1)

ax.text(0.22, 0.5, f"{only_0_clean}", ha="center", va="center", fontsize=16, color="white")
ax.text(0.78, 0.5, f"{only_1_clean}", ha="center", va="center", fontsize=16, color="white")
ax.text(0.5, 0.5, f"{overlap_clean}", ha="center", va="center", fontsize=16, color="white")

ax.text(0.22, 0.85, "Nur Kein Disaster", ha="center", fontsize=11, color=COLOR_CLASS_0)
ax.text(0.78, 0.85, "Nur Disaster", ha="center", fontsize=11, color=COLOR_CLASS_1)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Vokabular-Ueberschneidung zwischen den Klassen (ohne Stopwords)")

save_figure(fig, "../reports/figures/01_eda_vocab_venn.png")
plt.show()

## Erkenntnis: Vokabulargroesse & Ueberschneidung

**Vokabular waechst noch:** Die Heaps'-Law-Kurve zeigt kein klares Plateau
bei 7.613 Tweets (21.678 unique Woerter) — die Kurve flacht leicht ab
(typisches Potenzgesetz-Verhalten), ist aber noch klar steigend. Das deutet
darauf hin, dass mehr Trainingsdaten vermutlich weiterhin neues, bisher
ungesehenes Vokabular einbringen wuerden (relevant fuer die spaetere Frage:
"wuerde mehr Daten helfen?").

**Klassenspezifisches Vokabular ueberwiegt:** Von 21.678 Woertern insgesamt
sind nur 4.036 (~19 %) in beiden Klassen vertreten (nach Stopword-Bereinigung).
17.495 Woerter (~81 %) kommen nur in einer der beiden Klassen vor — das ist
ein positives Signal fuer ein wortbasiertes Modell (TF-IDF): Es gibt
substanzielle Mengen an klassenspezifischem Vokabular zum Trennen.

**Stopwords spielen kaum eine Rolle:** Nur 3,4 % der urspruenglichen
Ueberschneidung (140 von 4.176 Woertern) sind klassische Stopwords. Die
verbleibende Ueberschneidung besteht ueberwiegend aus gewoehnlichen
englischen Inhaltswoertern (z. B. "like", "people", "time"), die thematisch
neutral und in beiden Klassen erwartbar sind.

In [ ]:
# =============================================================================
# Zelle 11 – Lexikalische Trennbarkeit: Welche Woerter trennen die Klassen?
# =============================================================================
# Chi²-Test misst, wie stark ein Wort statistisch mit der Zielvariable
# zusammenhaengt (unabhaengig von reiner Worthaeufigkeit).

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
import numpy as np

vectorizer = CountVectorizer(stop_words="english", min_df=5)
X_counts = vectorizer.fit_transform(train["text"])
feature_names = np.array(vectorizer.get_feature_names_out())

chi2_scores, p_values = chi2(X_counts, train["target"])

# Richtung der Assoziation: ueberwiegt das Wort in Klasse 1 (Disaster) oder Klasse 0?
class1_mask = (train["target"] == 1).values
freq_class1 = np.asarray(X_counts[class1_mask].sum(axis=0)).flatten()
freq_class0 = np.asarray(X_counts[~class1_mask].sum(axis=0)).flatten()

# Normalisiert auf Klassengroesse, damit Vergleich fair ist (unterschiedliche Klassengroessen)
rate_class1 = freq_class1 / class1_mask.sum()
rate_class0 = freq_class0 / (~class1_mask).sum()
association = np.where(rate_class1 > rate_class0, "Disaster", "Kein Disaster")

chi2_df = pd.DataFrame({
    "word": feature_names,
    "chi2_score": chi2_scores,
    "p_value": p_values,
    "association": association,
    "freq_class0": freq_class0,
    "freq_class1": freq_class1,
})

top20 = chi2_df.sort_values("chi2_score", ascending=False).head(20)
print(top20[["word", "chi2_score", "association", "freq_class0", "freq_class1"]])

chi2_df.sort_values("chi2_score", ascending=False).to_csv("../reports/tables/01_chi2_top_words.csv", index=False)

# Plot: Top-20-Woerter, eingefaerbt nach Klassen-Assoziation
fig, ax = plt.subplots(figsize=(8, 8))
colors = [COLOR_CLASS_1 if a == "Disaster" else COLOR_CLASS_0 for a in top20["association"]]
ax.barh(top20["word"][::-1], top20["chi2_score"][::-1], color=colors[::-1])
ax.set_xlabel("Chi2-Score")
ax.set_title("Top 20 trennscharfe Woerter")

save_figure(fig, "../reports/figures/01_eda_chi2_top_words.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 11b – Top-15 Woerter je Klasse im direkten Vergleich
# =============================================================================

top15_disaster = chi2_df[chi2_df["association"]=="Disaster"].sort_values("chi2_score", ascending=False).head(15)
top15_kein = chi2_df[chi2_df["association"]=="Kein Disaster"].sort_values("chi2_score", ascending=False).head(15)

top15_kein.to_csv("../reports/tables/01_chi2_top_words_kein_disaster.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 7))

axes[0].barh(top15_kein["word"][::-1], top15_kein["chi2_score"][::-1], color=COLOR_CLASS_0)
axes[0].set_title("Top 15 'Kein Disaster'")
axes[0].set_xlabel("Chi2-Score")

axes[1].barh(top15_disaster["word"][::-1], top15_disaster["chi2_score"][::-1], color=COLOR_CLASS_1)
axes[1].set_title("Top 15 'Disaster'")
axes[1].set_xlabel("Chi2-Score")

# Gleiche X-Achsen-Skala fuer fairen visuellen Vergleich
max_score = max(top15_kein["chi2_score"].max(), top15_disaster["chi2_score"].max())
axes[0].set_xlim(0, max_score)
axes[1].set_xlim(0, max_score)

fig.suptitle("Trennschaerfe im Vergleich: Disaster-Vokabular deutlich konzentrierter")
fig.tight_layout()

save_figure(fig, "../reports/figures/01_eda_chi2_comparison.png")
plt.show()

## Erkenntnis: Lexikalische Trennbarkeit

**Disaster-Vokabular ist hoch spezifisch und fast exklusiv:** Woerter wie
`debris`, `hiroshima`, `mh370`, `typhoon` kommen nahezu ausschliesslich im
Disaster-Kontext vor (z. B. `debris`: 0x in Klasse 0, 50x in Klasse 1).
Hoechster Chi2-Score: `http` (266,6) — vermutlich Nachrichtenartikel-Links.

**'Kein Disaster' ist eine heterogene Sammelkategorie ohne gemeinsames
Thema:** Mittlerer Chi2-Score nur 3,33 (vs. 9,28 bei Disaster-Woertern),
hoechster Wert nur 53,1 (`body`). Es gibt kein vergleichbar starkes
"negatives Signalwort" — die Klasse umfasst thematisch alles, was kein
Disaster ist (Alltag, Beziehungen, Humor, Konsum).

**Konsequenz fuer die Modellierung:** Ein wortbasiertes Modell (TF-IDF)
wird vermutlich deutlich zuverlaessiger erkennen, WANN ein Tweet ein
Disaster behandelt (starke positive Signalwoerter vorhanden), als
zuverlaessig auszuschliessen, dass es KEINER ist (keine vergleichbar
starken negativen Signalwoerter). Erwartung: Fehlklassifikationen werden
sich vermutlich eher bei Klasse 0 haeufen (False Positives durch
metaphorische Nutzung von Disaster-Vokabular, z. B. "this song is fire"),
zu pruefen in Phase 6 (Fehleranalyse).

**Hinweis `http` als Top-Feature:** Bestaetigt den frueheren Befund zu
URL-Praesenz (66,4% vs. 41,4%) — die Tatsache, dass ein Tweet UEBERHAUPT
einen Link enthaelt, ist selbst ein starkes Signal. Spricht zusaetzlich
fuer `has_url` als eigenes Feature in Phase 3/4.

In [ ]:
# =============================================================================
# Zelle 13 – Label-Rauschen: gezielte Stichprobe ambiger Faelle (korrigiert)
# =============================================================================
# Korrektur: 'http' ausgeschlossen (strukturelles Merkmal, kein Bedeutungstraeger,
# bereits separat in Zelle 07b dokumentiert). Nicht-capturing Group behebt
# zusaetzlich die Regex-Warnung der Vorversion.

strong_disaster_words = chi2_df[chi2_df["association"]=="Disaster"] \
    .sort_values("chi2_score", ascending=False)
strong_disaster_words = strong_disaster_words[strong_disaster_words["word"] != "http"] \
    .head(15)["word"].tolist()

print("Verwendete Signalwoerter (ohne 'http'):", strong_disaster_words)

pattern = r"\b(?:" + "|".join(strong_disaster_words) + r")\b"
contains_strong_word = train["text"].str.lower().str.contains(pattern, regex=True, na=False)

ambiguous_candidates = train[contains_strong_word & (train["target"] == 0)]

print(f"\n{len(ambiguous_candidates)} potenziell ambige Faelle gefunden (echtes Disaster-Wort, Label=0)")
print("\nAlle Faelle zur manuellen Durchsicht:\n")
for _, row in ambiguous_candidates.iterrows():
    print(f"  [{row['target']}] {row['text']}")

ambiguous_candidates.to_csv("../reports/tables/01_ambiguous_label_candidates.csv", index=False)

## Erkenntnis: Label-Rauschen & semantische Ambiguitaet

Manuelle Durchsicht von 85 Tweets mit starkem Disaster-Vokabular bei Label=0
zeigt mehrere Ursachen fuer scheinbare Fehlklassifikation, keine Label-Fehler:

1. **Idiomatische/metaphorische Nutzung** ("I'm a disaster", "fires back",
   Songtexte) – Wortbedeutung loest sich vom Thema.
2. **Diskussion/Hypothetisches statt laufendes Ereignis** (Statistiken,
   Vorbereitungs-Tipps, PSA-Tweets) – Thema vorhanden, aber kein aktuelles
   Ereignis. Erfordert Diskursverstaendnis, nicht nur Wortpraesenz.
3. **Zu geringfuegig fuer "Disaster"** (z. B. Mini-Erdbeben M1.4) – Label
   vermutlich korrekt, Grenzfall der Definition.

**Kernaussage:** Die ueberwiegende Mehrheit der gepruefchen Faelle sind
KEINE Label-Fehler, sondern zeigen die strukturelle Grenze wortbasierter
Modelle (TF-IDF/BoW): Wortpraesenz ohne Kontext, Zeitform oder Diskursebene.
Das ist der konkrete, datengetriebene Beleg fuer den Mehrwert kontextueller
Modelle (Embeddings/Transformer, Bonus-Branch) gegenueber dem Standardansatz.

**Nebenfund:** Near-Duplicates (identischer Text, unterschiedliche Kurz-URL,
z. B. "Ted Cruz fires back..." 4x) werden von der exakten Duplikat-Pruefung
(Zelle 03) nicht erfasst – vermerkt, keine Aktion in dieser Phase.

In [ ]:
# =============================================================================
# Zelle 15 – Noise-Elemente: Mentions, Hashtags, Encoding-Artefakte
# =============================================================================

train["n_mentions"] = train["text"].str.count(r"@\w+")
train["n_hashtags"] = train["text"].str.count(r"#\w+")
train["has_mention"] = train["n_mentions"] > 0
train["has_hashtag"] = train["n_hashtags"] > 0

# Encoding-Mojibake: typische Muster fehlerhaft dekodierter Sonderzeichen
# (Windows-1252 als UTF-8 fehlinterpretiert -> Û, å, Ê, ÛÒ, ÛÏ etc.)
train["has_encoding_artifact"] = train["text"].str.contains(r"[ÛåÊ]", regex=True, na=False)

noise_summary = pd.DataFrame({
    "has_url_pct": train.groupby("target")["has_url"].mean() * 100,
    "has_mention_pct": train.groupby("target")["has_mention"].mean() * 100,
    "has_hashtag_pct": train.groupby("target")["has_hashtag"].mean() * 100,
    "has_encoding_artifact_pct": train.groupby("target")["has_encoding_artifact"].mean() * 100,
}).round(1)

print(noise_summary)
noise_summary.to_csv("../reports/tables/01_noise_elements_by_class.csv")

print(f"\nGesamt-Tweets mit Encoding-Artefakten: {train['has_encoding_artifact'].sum()} ({train['has_encoding_artifact'].mean()*100:.1f}%)")
print("\nBeispiele:")
for t in train[train["has_encoding_artifact"]]["text"].head(3):
    print(f"  {t}")

# Plot: alle vier Noise-Elemente im Vergleich nach Klasse
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(4)
width = 0.35
labels = ["URL", "Mention", "Hashtag", "Encoding-Artefakt"]

vals0 = noise_summary.loc[0].values
vals1 = noise_summary.loc[1].values

ax.bar(x - width/2, vals0, width, label="Kein Disaster", color=COLOR_CLASS_0)
ax.bar(x + width/2, vals1, width, label="Disaster", color=COLOR_CLASS_1)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Anteil Tweets (%)")
ax.set_title("Noise-Elemente nach Klasse")
ax.legend()

save_figure(fig, "../reports/figures/01_eda_noise_elements.png")
plt.show()

## Erkenntnis: Noise-Elemente (URLs, Mentions, Hashtags, Encoding)

**URL — Signal vs. Noise sind zu trennen:** Der hohe Chi2-Score von "http"
(266,6, Top-Wort) und die hohe URL-Praesenz (66,4% vs. 41,4%) beziehen sich
NUR auf das wiederkehrende Wort "http" selbst, nicht auf den Rest der URL
(z. B. "t.co/wDUEaj8Q4J"). Der URL-Pfad ist bei jedem Tweet zufaellig/
einzigartig und liefert kein generalisierbares Signal — er blaeht nur das
Vokabular auf. Konsequenz fuer Phase 2: URLs aus dem Fliesstext entfernen,
das bereits erfasste `has_url`-Feature (binaer) bleibt erhalten und ersetzt
die Information kompakter.

**Mentions verhalten sich umgekehrt zu URLs/Hashtags:** Haeufiger bei
"Kein Disaster" (30,9% vs. 20,4%) — passt zur Interpretation, dass
persoenliche/konversationelle Tweets sich eher an andere Nutzer richten,
waehrend Disaster-Tweets eher an ein allgemeines Publikum gerichtet sind
(News-Charakter).

**Encoding-Artefakte (Û, å, Ê) sind echtes, neutrales Rauschen:** Aehnliche
Haeufigkeit in beiden Klassen (7,9% vs. 10,1%), kein auffaelliger
Unterschied — stammt vermutlich aus fehlerhafter Zeichenkodierung
(Windows-1252 als UTF-8 fehlinterpretiert) bei der Datenerhebung. Kein
prädiktiver Wert, wird in Phase 2 normalisiert/entfernt.

**Geplante Preprocessing-Schritte (Phase 2), abgeleitet aus diesen Befunden:**
- URLs aus Text entfernen, `has_url` als separates Feature behalten
- Encoding-Artefakte normalisieren/entfernen
- Mentions/Hashtags: vorerst im Text belassen (moegliches Signal je nach
  Kontext), `has_mention`/`has_hashtag` zusaetzlich als Features verfuegbar

In [ ]:
# =============================================================================
# Zelle 17 – keyword: Tiefenanalyse (mit dekodierten %20-Zeichen)
# =============================================================================

# Korrektur: %20 ist URL-Encoding fuer Leerzeichen, vor der Analyse dekodieren
train["keyword"] = train["keyword"].str.replace("%20", " ", regex=False)

print("Anzahl eindeutiger keyword-Werte:", train["keyword"].nunique())
print("\nHaeufigste keywords:\n", train["keyword"].value_counts().head(10))

keyword_stats = train.groupby("keyword")["target"].agg(["mean", "count"]).rename(columns={"mean": "disaster_rate"})
keyword_stats = keyword_stats[keyword_stats["count"] >= 10].sort_values("disaster_rate", ascending=False)

print(f"\n{len(keyword_stats)} keywords mit >=10 Vorkommen")
print("\nTop 10 keywords mit hoechster Disaster-Rate:")
print(keyword_stats.head(10))
print("\nTop 10 keywords mit niedrigster Disaster-Rate:")
print(keyword_stats.tail(10))

keyword_stats.to_csv("../reports/tables/01_keyword_disaster_rate.csv")

combined = pd.concat([keyword_stats.tail(10), keyword_stats.head(10)])
base_rate = train["target"].mean()

fig, ax = plt.subplots(figsize=(8, 8))
colors = [COLOR_CLASS_0 if r < base_rate else COLOR_CLASS_1 for r in combined["disaster_rate"]]
ax.barh(combined.index, combined["disaster_rate"], color=colors)
ax.axvline(base_rate, color=COLOR_TEXT_MUTED, linestyle="--", label=f"Basisrate ({base_rate:.2f})")
ax.set_xlabel("Anteil Disaster (target=1)")
ax.set_title("Keywords: staerkste Disaster-Indikatoren (>=10 Vorkommen)")
ax.legend()

save_figure(fig, "../reports/figures/01_eda_keyword_disaster_rate.png")
plt.show()

In [ ]:
# =============================================================================
# Zelle 17b – location: Tiefenanalyse (Missingness bereits bekannt: 33,3%)
# =============================================================================

print("Anzahl eindeutiger location-Werte:", train["location"].nunique())
print("\nHaeufigste location-Eintraege:\n", train["location"].value_counts().head(20))

# Disaster-Rate fuer die haeufigsten Locations (nur ausreichend Vorkommen)
location_stats = train.groupby("location")["target"].agg(["mean", "count"]).rename(columns={"mean": "disaster_rate"})
location_stats_filtered = location_stats[location_stats["count"] >= 10].sort_values("count", ascending=False)

print(f"\n{len(location_stats_filtered)} locations mit >=10 Vorkommen")
print(location_stats_filtered.head(15))

location_stats.to_csv("../reports/tables/01_location_disaster_rate.csv")

In [ ]:
# =============================================================================
# Zelle 17c – Kardinalitaet: keyword vs. location im Vergleich
# =============================================================================

def cardinality_stats(series, name):
    non_null = series.dropna()
    n_unique = non_null.nunique()
    n_total = len(non_null)
    ratio = n_unique / n_total
    singleton_share = (non_null.value_counts() == 1).mean() * 100
    return {
        "spalte": name,
        "eindeutige_werte": n_unique,
        "nicht_fehlende_werte": n_total,
        "kardinalitaets_ratio": round(ratio, 3),
        "anteil_einzelvorkommen_pct": round(singleton_share, 1),
    }

cardinality_comparison = pd.DataFrame([
    cardinality_stats(train["keyword"], "keyword"),
    cardinality_stats(train["location"], "location"),
])
print(cardinality_comparison)

cardinality_comparison.to_csv("../reports/tables/01_cardinality_comparison.csv", index=False)

## Erkenntnis: keyword & location

**keyword — gut nutzbares Feature:**
- 221 eindeutige Werte, Kardinalitaets-Ratio 0,029 (jeder Wert kommt im
  Schnitt ~34x vor) — ausreichend Wiederholung fuer stabile Statistik
- 0% Einzelvorkommen — jedes keyword hat mehrfache Beispiele
- Disaster-Rate pro keyword reicht von 0% (z. B. "aftershock") bis 100%
  (z. B. "debris", "derailment", "wreckage") — bimodales Muster,
  kaum Werte nahe der Basisrate (0,43)
- Interpretation: keyword ist vermutlich der urspruengliche Such-/
  Scraping-Begriff; die Disaster-Rate zeigt, wie oft ein Wort in der
  Praxis woertlich vs. metaphorisch verwendet wird — bestaetigt und
  quantifiziert den Kontext-Befund aus Zelle 13/14
- Geringe Fehlquote (0,8%)
- **Entscheidung: keyword wird als Feature in Phase 3/4 beruecksichtigt**

**location — nicht nutzbar in Rohform:**
- 3.341 eindeutige Werte auf 5.080 nicht-fehlende Zeilen, Kardinalitaets-
  Ratio 0,658 (jeder Wert kommt im Schnitt nur ~1,5x vor)
- 84,3% aller eindeutigen Werte kommen nur einmal vor — keine stabile
  Statistik pro Wert moeglich
- Zusaetzlich: hohe Fehlquote (33,3%) und Format-Inkonsistenz
  (z. B. "USA" / "United States" / "California, USA" fuer denselben Ort)
- **Entscheidung: location wird NICHT als Feature im Standardprojekt
  verwendet.** Aufbereitungsaufwand (Geocoding, Normalisierung) steht in
  keinem Verhaeltnis zum erwartbaren Nutzen bei dieser Datenqualitaet.
  Spalte bleibt im Datensatz erhalten (keine Loeschung), nur ungenutzt.

In [ ]:
# =============================================================================
# Zelle 19 – Baseline: triviale Klassifikatoren als untere Leistungsgrenze
# =============================================================================
# Zeigt, was ein Modell OHNE jegliches Lernen erreichen wuerde — jede
# spaetere Modellierung muss sich daran messen, nicht an 0.

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_dummy = train[["text"]]  # Platzhalter, Dummy-Classifier ignoriert Features
y = train["target"]

strategies = ["most_frequent", "stratified", "uniform"]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_results = []
for strategy in strategies:
    clf = DummyClassifier(strategy=strategy, random_state=42)
    f1_scores = cross_val_score(clf, X_dummy, y, cv=skf, scoring="f1")
    baseline_results.append({
        "strategy": strategy,
        "f1_mean": f1_scores.mean(),
        "f1_std": f1_scores.std(),
    })

baseline_df = pd.DataFrame(baseline_results)
print(baseline_df)

baseline_df.to_csv("../reports/tables/01_baseline_dummy_scores.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(baseline_df["strategy"], baseline_df["f1_mean"],
               yerr=baseline_df["f1_std"], capsize=5,
               color=[COLOR_BLUE, COLOR_GOLD, COLOR_GREEN])
ax.set_ylabel("F1-Score (5-Fold CV Mittelwert)")
ax.set_title("Baseline: Dummy-Klassifikatoren (untere Leistungsgrenze)")
ax.set_ylim(0, 1)

for bar, val in zip(bars, baseline_df["f1_mean"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.3f}", ha="center", color="white")

save_figure(fig, "../reports/figures/01_eda_baseline_dummy.png")
plt.show()

## Erkenntnis: Baseline (untere Leistungsgrenze)

| Strategie | F1-Score | Bedeutung |
|---|---|---|
| most_frequent | 0,000 | Immer "kein Disaster" — null Disaster-Erkennung |
| stratified | 0,445 | Zufall, proportional zur Klassenverteilung |
| uniform | 0,468 | Reiner 50/50-Zufall |

**Wichtigste Erkenntnis:** most_frequent erreicht trotz ~57% Accuracy einen
F1-Score von 0 — bestaetigt eindeutig, warum Accuracy als Metrik hier
ungeeignet waere und F1 die richtige Wahl ist (bereits in Zelle 06
begruendet, hier empirisch verifiziert).

**Referenzpunkt fuer Phase 4:** Jedes trainierte Modell muss F1 > 0,47
(beste Dummy-Baseline) klar uebertreffen, sonst hat es nichts Sinnvolles
gelernt. Realistischer SOTA-Referenzwert liegt bei F1 ~0,83-0,85
(BERT-Klasse) — die Baseline zeigt den unteren Rand des Spektrums,
der unbereinigte naive Chi2-Befund (z. B. "debris" 100% Disaster-Rate)
deutet bereits darauf hin, dass ein einfaches Wort-basiertes Modell
deutlich ueber dieser Baseline liegen sollte.

In [ ]:
# =============================================================================
# Zelle 21 – Bereinigung: Duplikate & Label-Konflikte entfernen,
# bereinigten Datensatz fuer Phase 2 speichern
# =============================================================================

print("Shape vor Bereinigung:", train.shape)

# Entfernen: Label-Konflikte (echtes Rauschen) + harmlose exakte Duplikate
train_clean = train[~train["is_label_conflict"] & ~train["is_exact_duplicate"]].copy()

print("Shape nach Bereinigung:", train_clean.shape)
print("Entfernte Zeilen:", train.shape[0] - train_clean.shape[0])

# Aufgeraeumte Hilfsspalten entfernen, die nur fuer die EDA-Filterung
# gebraucht wurden (nach der Bereinigung uninformativ, da nun immer False)
train_clean = train_clean.drop(columns=["is_label_conflict", "is_exact_duplicate"])

# 'bin' war nur temporaer fuer die Laengen-Crosstab-Analyse, nicht persistieren
if "bin" in train_clean.columns:
    train_clean = train_clean.drop(columns=["bin"])

print("\nVerbleibende Spalten:", list(train_clean.columns))
print("\nNeue Klassenverteilung nach Bereinigung:")
print(train_clean["target"].value_counts(normalize=True).round(3))

# Speichern fuer Phase 2
train_clean.to_csv("../data/processed/train_clean.csv", index=False)
print(f"\nGespeichert: data/processed/train_clean.csv ({len(train_clean)} Zeilen)")

# Abschluss: Notebook 01 — EDA & Datenqualitaet

## Datensatz-Status

- **Ausgangslage:** 7.613 Tweets, 5 Spalten (id, keyword, location, text, target)
- **Nach Bereinigung:** 7.434 Tweets (179 entfernt: 55 Label-Konflikte +
  124 exakte Duplikate)
- **Neue Klassenverteilung:** 57,6% kein Disaster / 42,4% Disaster
  (kaum veraendert ggue. Ausgangslage)
- **Gespeichert:** `data/processed/train_clean.csv`
- **Neue Features:** char_count, word_count, has_url, n_mentions,
  n_hashtags, has_mention, has_hashtag, has_encoding_artifact

## Kernerkenntnisse

1. **Datenqualitaet:** 18 echte Label-Konflikte gefunden und entfernt;
   ueberwiegender Teil der "ambigen" Faelle erwies sich jedoch als
   Kontext-/Metapher-Phaenomen, nicht als Label-Fehler

2. **Klassenbalance:** Moderates Ungleichgewicht (57/43) — rechtfertigt
   F1 statt Accuracy, aber keine extreme Schieflage, die Class Weighting
   zwingend erfordern wuerde

3. **Trennbarkeit:** Disaster-Vokabular ist hoch spezifisch und
   konzentriert (z. B. "debris" 100% exklusiv), "Kein Disaster" ist eine
   heterogene Sammelkategorie ohne vergleichbar starkes Gegen-Vokabular —
   ein wortbasiertes Modell wird vermutlich besser darin sein, Disaster
   zu ERKENNEN als sicher AUSZUSCHLIESSEN

4. **Strukturelle Modellgrenze identifiziert:** Manuelle Durchsicht
   zeigte zahlreiche Faelle, in denen Disaster-Vokabular metaphorisch
   oder im hypothetischen/diskursiven Kontext verwendet wird (z. B.
   "I'm a disaster", "Practice your fire escape plan"). Das ist die
   konkrete, datengetriebene Begruendung fuer den Mehrwert kontextueller
   Modelle (Embeddings/Transformer) im Bonus-Branch.

5. **Feature-Qualitaet sehr unterschiedlich:**
   - `keyword`: stark nutzbar (Kardinalitaets-Ratio 0,029, bimodale
     Disaster-Rate 0%-100%)
   - `location`: nicht nutzbar in Rohform (Ratio 0,658, 84,3%
     Einzelvorkommen, hohe Fehlquote) — wird im Standardprojekt
     nicht als Feature verwendet
   - `has_url`: starkes Signal (66,4% vs. 41,4%), aber URL-Inhalt selbst
     ist Noise — nur die Praesenz ist informativ

6. **Baseline etabliert:** F1 zwischen 0,000 (most_frequent) und 0,468
   (uniform) — Referenzpunkt fuer alle kommenden Modelle (Phase 4+).
   SOTA-Referenz bleibt F1 ~0,83-0,85 (realistischer BERT-Wert).

## Konsequenzen fuer Phase 2 (Preprocessing)

- URLs aus Fliesstext entfernen, has_url-Feature behalten
- Encoding-Artefakte (Û, å, Ê) normalisieren
- keyword (%20-bereinigt) als Feature-Kandidat uebernehmen
- location nicht weiterverwenden
- Stemming UND Lemmatisierung parallel anlegen (Spalten anhaengen,
  Vergleich folgt)

## Offene Punkte / spaeter zu pruefen

- Near-Duplicates (identischer Text, unterschiedliche Kurz-URL) wurden
  beobachtet, aber nicht systematisch erfasst — ggf. in Phase 6 relevant